# Autoregressive (AR) Models

An **autoregressive model** of order $p$, written AR($p$), expresses the
current value of a time series as a linear combination of its own past
values plus a white-noise error term:

$$
y_t = c + \phi_1 y_{t-1} + \phi_2 y_{t-2} + \cdots + \phi_p y_{t-p} + \varepsilon_t
$$

where $\varepsilon_t$ is white noise and $\phi_1, \ldots, \phi_p$ are the
autoregressive coefficients.

The name "autoregressive" means the model **regresses $y_t$ on its own
lagged values** — it is a regression of the variable against itself.

This notebook covers:
1. AR(1) and AR(2) models — theory, stationarity conditions, and simulation
2. Backshift notation
3. Using the PACF to identify the order $p$
4. Fitting AR models with the modern `AutoReg` API in statsmodels
5. Application to real data

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")
rng = np.random.default_rng(42)

---
## 1. The AR(1) Model

The simplest autoregressive model uses only **one lag**:

$$
y_t = c + \phi_1 y_{t-1} + \varepsilon_t
$$

Key properties:
- If $|\phi_1| < 1$, the process is **stationary** — it fluctuates around
  the mean $\mu = c / (1 - \phi_1)$.
- If $\phi_1 = 1$, the process is a **random walk** (non-stationary).
- $\phi_1$ controls the **memory** of the process:
  - $\phi_1 > 0$: positive autocorrelation — high values tend to follow high values
  - $\phi_1 < 0$: negative autocorrelation — values alternate above and below the mean
  - $\phi_1 \approx 0$: essentially white noise

In [ ]:
def simulate_ar1(phi1, c=0, n=300, seed=42):
    """Simulate an AR(1) process: y_t = c + phi1 * y_{t-1} + eps_t."""
    rng_local = np.random.default_rng(seed)
    eps = rng_local.standard_normal(n)
    y = np.zeros(n)
    for t in range(1, n):
        y[t] = c + phi1 * y[t - 1] + eps[t]
    return y


# Simulate AR(1) with different phi values
phi_values = [0.9, 0.5, -0.5, -0.9]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, phi in zip(axes.flat, phi_values):
    y = simulate_ar1(phi)
    ax.plot(y, linewidth=0.8)
    ax.axhline(0, color="grey", linestyle="--", alpha=0.5)
    ax.set_title(f"AR(1) with $\\phi_1 = {phi}$", fontsize=12)
    ax.set_xlabel("t")

plt.suptitle("Simulated AR(1) Processes", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("phi=0.9:  Strong positive autocorrelation — smooth, slow-wandering series.")
print("phi=0.5:  Moderate positive autocorrelation — less persistent.")
print("phi=-0.5: Moderate negative autocorrelation — values alternate.")
print("phi=-0.9: Strong negative autocorrelation — rapid oscillation.")

### ACF and PACF of AR(1)

For an AR(1) process:
- The **ACF** decays exponentially (geometric decay): $\rho_k = \phi_1^k$
- The **PACF** has a single spike at lag 1 and cuts off to zero after that

This PACF "cutoff" pattern is the key diagnostic for identifying AR order.

In [ ]:
y_ar1 = simulate_ar1(phi1=0.7, n=500)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(y_ar1, lags=20, ax=axes[0], title="ACF of AR(1), $\\phi_1=0.7$")
plot_pacf(y_ar1, lags=20, ax=axes[1], title="PACF of AR(1), $\\phi_1=0.7$", method="ywm")
plt.tight_layout()
plt.show()

print("ACF: exponential decay — autocorrelation diminishes gradually.")
print("PACF: single significant spike at lag 1, then cuts off.")
print("This PACF pattern tells us the true order is p=1.")

---
## 2. The AR(2) Model

$$
y_t = c + \phi_1 y_{t-1} + \phi_2 y_{t-2} + \varepsilon_t
$$

Stationarity conditions for AR(2) are more complex than AR(1).  The process
is stationary if and only if:
- $\phi_2 + \phi_1 < 1$
- $\phi_2 - \phi_1 < 1$
- $|\phi_2| < 1$

Depending on the signs and magnitudes of $\phi_1$ and $\phi_2$, the ACF
can show damped exponential decay or damped sinusoidal decay.

In [ ]:
def simulate_ar2(phi1, phi2, c=0, n=300, seed=42):
    """Simulate an AR(2) process."""
    rng_local = np.random.default_rng(seed)
    eps = rng_local.standard_normal(n)
    y = np.zeros(n)
    for t in range(2, n):
        y[t] = c + phi1 * y[t - 1] + phi2 * y[t - 2] + eps[t]
    return y


# Two examples: one with damped exponential, one with damped oscillation
y_ar2a = simulate_ar2(phi1=0.5, phi2=0.3, n=500)
y_ar2b = simulate_ar2(phi1=0.3, phi2=-0.5, n=500)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(y_ar2a[:200], linewidth=0.8)
axes[0].set_title("AR(2): $\\phi_1=0.5, \\phi_2=0.3$ (damped exponential)")
axes[1].plot(y_ar2b[:200], linewidth=0.8)
axes[1].set_title("AR(2): $\\phi_1=0.3, \\phi_2=-0.5$ (damped oscillation)")
for ax in axes:
    ax.set_xlabel("t")
    ax.axhline(0, color="grey", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# ACF and PACF of AR(2)
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

plot_acf(y_ar2a, lags=20, ax=axes[0, 0], title="ACF — AR(2): $\\phi_1=0.5, \\phi_2=0.3$")
plot_pacf(y_ar2a, lags=20, ax=axes[0, 1], title="PACF — AR(2): $\\phi_1=0.5, \\phi_2=0.3$", method="ywm")
plot_acf(y_ar2b, lags=20, ax=axes[1, 0], title="ACF — AR(2): $\\phi_1=0.3, \\phi_2=-0.5$")
plot_pacf(y_ar2b, lags=20, ax=axes[1, 1], title="PACF — AR(2): $\\phi_1=0.3, \\phi_2=-0.5$", method="ywm")

plt.tight_layout()
plt.show()

print("Key pattern: PACF has significant spikes at lags 1 and 2, then cuts off.")
print("This tells us the true order is p=2.")
print("The ACF decays gradually (exponential or sinusoidal depending on coefficients).")

---
## 3. Backshift Notation

The **backshift operator** $B$ shifts a time series back by one period:

$$
B y_t = y_{t-1}, \quad B^2 y_t = y_{t-2}, \quad \ldots, \quad B^k y_t = y_{t-k}
$$

Using backshift notation, an AR($p$) model can be written compactly as:

$$
(1 - \phi_1 B - \phi_2 B^2 - \cdots - \phi_p B^p) y_t = c + \varepsilon_t
$$

or equivalently: $\Phi(B) y_t = c + \varepsilon_t$, where
$\Phi(B) = 1 - \phi_1 B - \phi_2 B^2 - \cdots - \phi_p B^p$ is the
**AR characteristic polynomial**.

The stationarity condition is that all roots of $\Phi(B) = 0$ must lie
**outside the unit circle** in the complex plane.

### Examples

- AR(1): $(1 - \phi_1 B) y_t = c + \varepsilon_t$
- AR(2): $(1 - \phi_1 B - \phi_2 B^2) y_t = c + \varepsilon_t$

This notation will become essential when we combine AR with differencing
and MA to form ARIMA models.

In [ ]:
# Verify stationarity via characteristic roots for our AR(2) examples

def check_ar_stationarity(phi_coeffs):
    """Check stationarity by finding roots of the AR characteristic polynomial.

    The polynomial is: 1 - phi_1*z - phi_2*z^2 - ...
    Rewritten as: z^p - phi_1*z^{p-1} - ... (multiply through and find roots).
    Stationarity requires all roots to have modulus > 1.
    """
    # Coefficients for np.roots: [1, -phi_1, -phi_2, ...]
    poly = [1] + [-phi for phi in phi_coeffs]
    roots = np.roots(poly)
    moduli = np.abs(roots)
    is_stationary = np.all(moduli > 1)
    return roots, moduli, is_stationary


# Example 1: AR(2) with phi1=0.5, phi2=0.3
roots, moduli, stationary = check_ar_stationarity([0.5, 0.3])
print("AR(2): phi1=0.5, phi2=0.3")
print(f"  Roots: {roots}")
print(f"  |roots|: {moduli.round(4)}")
print(f"  Stationary: {stationary}")

print()

# Example 2: AR(1) with phi1=0.7
roots, moduli, stationary = check_ar_stationarity([0.7])
print("AR(1): phi1=0.7")
print(f"  Root: {roots}")
print(f"  |root|: {moduli.round(4)}")
print(f"  Stationary: {stationary}")

print()

# Example 3: Non-stationary — phi1=1.0 (unit root / random walk)
roots, moduli, stationary = check_ar_stationarity([1.0])
print("AR(1): phi1=1.0 (random walk)")
print(f"  Root: {roots}")
print(f"  |root|: {moduli.round(4)}")
print(f"  Stationary: {stationary}  (unit root — not stationary)")

---
## 4. PACF Identifies the AR Order

The **partial autocorrelation function (PACF)** at lag $k$ measures the
correlation between $y_t$ and $y_{t-k}$ **after removing** the linear
effects of the intermediate lags $y_{t-1}, \ldots, y_{t-k+1}$.

For a pure AR($p$) process:
- PACF is **nonzero** at lags $1, 2, \ldots, p$
- PACF is **zero** for all lags $> p$ (it "cuts off" after lag $p$)

This is the primary tool for selecting the AR order.

| Model | ACF pattern | PACF pattern |
|-------|-------------|---------------|
| AR($p$) | Exponential/sinusoidal decay | Cuts off after lag $p$ |
| MA($q$) | Cuts off after lag $q$ | Exponential/sinusoidal decay |
| ARMA($p$,$q$) | Exponential decay | Exponential decay |

In [ ]:
# Demonstrate PACF cutoff for AR(1), AR(2), AR(3)
y_ar1 = simulate_ar1(phi1=0.6, n=1000, seed=10)
y_ar2 = simulate_ar2(phi1=0.4, phi2=0.3, n=1000, seed=10)

# AR(3) simulation
def simulate_ar3(phi1, phi2, phi3, c=0, n=300, seed=42):
    rng_local = np.random.default_rng(seed)
    eps = rng_local.standard_normal(n)
    y = np.zeros(n)
    for t in range(3, n):
        y[t] = c + phi1 * y[t-1] + phi2 * y[t-2] + phi3 * y[t-3] + eps[t]
    return y

y_ar3 = simulate_ar3(0.3, 0.2, 0.15, n=1000, seed=10)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, y, title in zip(axes, [y_ar1, y_ar2, y_ar3],
                         ["AR(1): $\\phi_1=0.6$",
                          "AR(2): $\\phi_1=0.4, \\phi_2=0.3$",
                          "AR(3): $\\phi_1=0.3, \\phi_2=0.2, \\phi_3=0.15$"]):
    plot_pacf(y, lags=15, ax=ax, title=f"PACF — {title}", method="ywm")

plt.tight_layout()
plt.show()

print("AR(1) PACF: cuts off after lag 1.")
print("AR(2) PACF: cuts off after lag 2.")
print("AR(3) PACF: cuts off after lag 3.")
print("\nThe PACF cutoff directly reveals the true AR order.")

---
## 5. Fitting AR Models with `AutoReg`

The modern statsmodels API for pure AR models is `AutoReg` (NOT the old,
removed `AR` class).

```python
from statsmodels.tsa.ar_model import AutoReg
model = AutoReg(endog, lags=p)
result = model.fit()
```

Key methods:
- `result.summary()` — parameter estimates, standard errors, AIC/BIC
- `result.predict(start, end)` — in-sample and out-of-sample predictions
- `result.params` — coefficient values

In [ ]:
# Fit AutoReg to simulated AR(2) data to verify we recover the true parameters
y_sim = simulate_ar2(phi1=0.5, phi2=0.3, n=500, seed=99)
y_series = pd.Series(y_sim, name="y")

model = AutoReg(y_series, lags=2)
result = model.fit()
print(result.summary())

In [ ]:
# Compare estimated vs true parameters
print("True parameters:")
print(f"  phi_1 = 0.5")
print(f"  phi_2 = 0.3")
print(f"  c     = 0.0")
print()
print("Estimated parameters:")
print(f"  phi_1 = {result.params['y.L1']:.4f}")
print(f"  phi_2 = {result.params['y.L2']:.4f}")
print(f"  c     = {result.params['const']:.4f}")
print()
print("With 500 observations, the estimates are close to the true values.")

In [ ]:
# Predictions from the fitted model
# In-sample fitted values
fitted = result.fittedvalues

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(y_series[:100], label="Actual", alpha=0.7)
ax.plot(fitted[:100], label="Fitted (AR(2))", alpha=0.7, linestyle="--")
ax.set_title("AR(2) — Actual vs Fitted Values (first 100 obs)")
ax.legend()
ax.set_xlabel("t")
plt.tight_layout()
plt.show()

In [ ]:
# Out-of-sample prediction
forecast_steps = 24
predictions = result.predict(start=len(y_series), end=len(y_series) + forecast_steps - 1)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(y_series.iloc[-100:], label="Actual (last 100)")
ax.plot(predictions, label="Forecast (24 steps)", color="tab:red", linestyle="--")
ax.axvline(len(y_series) - 1, color="grey", linestyle=":", alpha=0.7)
ax.set_title("AR(2) — Out-of-Sample Forecast")
ax.legend()
plt.tight_layout()
plt.show()

print("The forecast converges to the process mean — a fundamental property")
print("of stationary AR models: long-horizon forecasts revert to the mean.")

### Selecting the lag order

How do we choose $p$ in practice?  We can:
1. Look at the PACF cutoff
2. Use information criteria (AIC, BIC) — fit models for $p = 1, 2, \ldots$
   and pick the one with the lowest AIC or BIC

In [ ]:
# Compare AIC/BIC for different lag orders on simulated AR(2) data
results_table = []
for p in range(1, 8):
    mod = AutoReg(y_series, lags=p).fit()
    results_table.append({"p": p, "AIC": round(mod.aic, 2), "BIC": round(mod.bic, 2)})

ic_df = pd.DataFrame(results_table)
print(ic_df.to_string(index=False))
print(f"\nBest by AIC: p={ic_df.loc[ic_df['AIC'].idxmin(), 'p']}")
print(f"Best by BIC: p={ic_df.loc[ic_df['BIC'].idxmin(), 'p']}")
print("\nBoth correctly identify p=2 as the best order.")

---
## 6. Application to Real Data

Let us apply AR modelling to the **Daily Total Female Births** dataset.
This is a roughly stationary daily series (no strong trend or seasonality),
making it a good candidate for a pure AR model.

In [ ]:
births = pd.read_csv(
    DATA_DIR / "DailyTotalFemaleBirths.csv",
    index_col="Date",
    parse_dates=True,
)
births.columns = ["Births"]

print(f"Shape: {births.shape}")
print(f"Date range: {births.index[0].date()} to {births.index[-1].date()}")
births.head()

In [ ]:
fig, ax = plt.subplots()
ax.plot(births["Births"], linewidth=0.8)
ax.set_title("Daily Total Female Births — 1959")
ax.set_ylabel("Number of Births")
ax.set_xlabel("Date")
plt.tight_layout()
plt.show()

print(f"Mean: {births['Births'].mean():.1f}")
print(f"Std:  {births['Births'].std():.1f}")
print("\nThe series looks roughly stationary — fluctuations around a constant level.")

In [ ]:
# ACF and PACF of births data
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(births["Births"], lags=30, ax=axes[0], title="ACF — Daily Births")
plot_pacf(births["Births"], lags=30, ax=axes[1], title="PACF — Daily Births", method="ywm")
plt.tight_layout()
plt.show()

print("PACF shows significant spikes — suggests a low-order AR model may be appropriate.")

In [ ]:
# Train/test split
TRAIN_END = 335
births_train = births.iloc[:TRAIN_END]
births_test = births.iloc[TRAIN_END:]

print(f"Train: {len(births_train)} days")
print(f"Test:  {len(births_test)} days")

In [ ]:
# Select lag order using AIC/BIC
results_table = []
for p in range(1, 15):
    mod = AutoReg(births_train["Births"], lags=p).fit()
    results_table.append({"p": p, "AIC": round(mod.aic, 2), "BIC": round(mod.bic, 2)})

ic_births = pd.DataFrame(results_table)
print(ic_births.to_string(index=False))

best_aic = ic_births.loc[ic_births["AIC"].idxmin()]
best_bic = ic_births.loc[ic_births["BIC"].idxmin()]
print(f"\nBest by AIC: p={int(best_aic['p'])} (AIC={best_aic['AIC']})")
print(f"Best by BIC: p={int(best_bic['p'])} (BIC={best_bic['BIC']})")

In [ ]:
# Fit the best model
best_p = int(best_bic["p"])
ar_model = AutoReg(births_train["Births"], lags=best_p)
ar_result = ar_model.fit()
print(ar_result.summary())

In [ ]:
# Forecast on the test set
ar_forecast = ar_result.predict(
    start=len(births_train),
    end=len(births_train) + len(births_test) - 1,
)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(births_train["Births"].iloc[-60:], label="Train (last 60 days)", color="black", alpha=0.5)
ax.plot(births_test["Births"], label="Actual (test)", color="tab:blue", linewidth=2)
ax.plot(births_test.index, ar_forecast.values, label=f"AR({best_p}) forecast",
        color="tab:red", linestyle="--")
ax.axvline(births_test.index[0], color="grey", linestyle=":", alpha=0.7)
ax.set_title(f"AR({best_p}) Forecast — Daily Births")
ax.set_ylabel("Births")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate forecast accuracy
from sklearn.metrics import mean_absolute_error, mean_squared_error

actual = births_test["Births"].values
pred = ar_forecast.values

mae = mean_absolute_error(actual, pred)
rmse = np.sqrt(mean_squared_error(actual, pred))

print(f"AR({best_p}) on Daily Births (test set):")
print(f"  MAE:  {mae:.2f}")
print(f"  RMSE: {rmse:.2f}")
print()
print("Note: the AR forecast quickly converges to the process mean for")
print("multi-step-ahead forecasts — this is expected for stationary data.")

---
## 7. Residual Diagnostics for the AR Model

In [ ]:
residuals = ar_result.resid

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Time plot of residuals
axes[0, 0].plot(residuals, linewidth=0.8)
axes[0, 0].axhline(0, color="red", linestyle="--", alpha=0.5)
axes[0, 0].set_title("Residuals over Time")

# Histogram
axes[0, 1].hist(residuals, bins=25, edgecolor="black", alpha=0.7, density=True)
axes[0, 1].set_title("Residual Distribution")

# ACF of residuals
plot_acf(residuals, lags=20, ax=axes[1, 0], title="ACF of Residuals")

# PACF of residuals
plot_pacf(residuals, lags=20, ax=axes[1, 1], title="PACF of Residuals", method="ywm")

plt.suptitle(f"AR({best_p}) Residual Diagnostics", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"Residual mean: {residuals.mean():.4f} (should be near zero)")
print(f"Residual std:  {residuals.std():.4f}")

In [ ]:
# Ljung-Box test on residuals
from statsmodels.stats.diagnostic import acorr_ljungbox

lb_test = acorr_ljungbox(residuals, lags=[10, 15, 20], return_df=True)
print("Ljung-Box test on AR residuals:")
print(lb_test)
print()
print("p-values > 0.05 indicate residuals are consistent with white noise.")

---
## Summary

- An **AR($p$)** model regresses $y_t$ on its own past $p$ values:
  $y_t = c + \phi_1 y_{t-1} + \cdots + \phi_p y_{t-p} + \varepsilon_t$.
- **Stationarity** requires $|\phi_1| < 1$ for AR(1); for higher orders,
  all roots of the characteristic polynomial must lie outside the unit circle.
- **Backshift notation** $\Phi(B) y_t = c + \varepsilon_t$ provides a compact
  representation.
- The **PACF cuts off after lag $p$** for a pure AR($p$) process — this is
  the primary diagnostic for order selection.
- Use `statsmodels.tsa.ar_model.AutoReg` (NOT the old `AR` class) for fitting.
- AR models are best suited for **stationary** data with autocorrelation
  structure.  For non-stationary data, we need differencing (the "I" in ARIMA).

In [ ]:
print("Next notebook: MA (Moving Average) models — the other building block of ARIMA.")